# 相机对齐(只看机器人)

在 **Isaac 电脑**上运行,内核选 **`resip`**。渲染时会调用 `isaaclab` 环境。

**思路**:真机和仿真摆成**同一组关节角**,只比较画面里的**机器人**,零件不用摆。
- **前视相机**:调外参(位置、俯角、偏航),直到仿真里的 FR3 和真实 FR3 重合。
- **腕部相机**:不调。直接用 FurnitureBench 支架的安装参数(手爪 −x 侧 5 cm、沿手指方向 4 cm、内倾 20°),以及 FR3 手爪模型。叠图只用来检查。
- 渲染用 Isaac Sim 自带的 FR3 模型(`--robot fr3`),仿真里的零件和挡条全部移出画面。

| 步骤 | 在哪台电脑 | 做什么 |
|---|---|---|
| A | **机器人电脑** | 移到参考姿态,读实际关节角和夹爪宽度,拍两张照片,读相机内参 |
| 1–2 | Isaac 电脑 | 配置;生成参考 pkl,写入内参 |
| 3–5 | Isaac 电脑 | 设前视外参 → 渲染 → 对比。**3–5 循环**,直到达标 |
| 6 | Isaac 电脑 | 保存基准 |

## A. 在机器人电脑上做(单独完成)

前提:已经在 Desk 里更新了末端负载,算上腕部相机和支架。不更新也能对齐,因为这里用的是实测关节角,但采集数据前必须更新。

1. 把本仓库的 `scratch/read_real_state.py` 拷到机器人电脑的 `~/ResiP/scratch/`。
2. 终端 1:启动 Polymetis 服务器。
   ```bash
   conda activate polymetis-local
   launch_robot.py robot_client=franka_hardware robot_client.executable_cfg.robot_ip=172.16.0.2 use_real_time=false
   ```
   如果装了夹爪,再开一个终端启动夹爪服务器(`launch_gripper.py gripper=franka_hand gripper.executable_cfg.robot_ip=172.16.0.2`)。
3. 终端 2:手放急停,运行下面的命令。它会移到参考姿态、等 5 s,然后读关节角和夹爪宽度、拍照、读内参。
   ```bash
   cd ~/ResiP && conda activate polymetis-local
   python scratch/read_real_state.py            # 没有夹爪服务器时加 --gripper-width 0.08(两指尖间距,米)
   ```
   桌上的零件不影响对齐,清空或保留都可以。注意不要挡住机器人。
4. 把机器人电脑上 `scratch/align/` 里的 `real_state.json`、`real_front.png`、`real_wrist.png` 拷到 Isaac 电脑的 `~/ResiP/scratch/align/`。

**每次重新拍照都要重做 2–5 步。** 前视外参调好之后,不需要再回到机器人电脑。

## 1. 配置(每次打开 notebook 先运行)

In [ ]:
import os, sys, re, json, glob, shutil, pickle, subprocess
from pathlib import Path
import numpy as np, cv2, matplotlib.pyplot as plt
from scipy import ndimage
from scipy.spatial.transform import Rotation as R

REPO = Path.home() / "ResiP"; os.chdir(REPO); sys.path.insert(0, str(REPO))
from src.visualization.render_mp4 import unpickle_data

ALIGN = REPO / "scratch" / "align"              # real_state.json / real_front.png / real_wrist.png 放这里
LOAD, OUT = ALIGN / "render_in", ALIGN / "render_out"
TEMPLATE = ALIGN / "template.pkl"               # 任意一条 one_leg rollout pkl,只借用它的数据格式
RR = REPO / "src" / "sim2real" / "isaac_lab_rerender.py"
ISAAC_PY = Path.home() / "miniconda3" / "envs" / "isaaclab" / "bin" / "python"
BASE_TO_WORLD = np.array([-0.30, 0.0, 0.415])   # 渲染器世界系 = 机器人基座系 + 这个
FLIP = np.diag([1, -1, -1])                     # OpenCV 相机系 <-> USD 相机系
if not TEMPLATE.exists() and (ALIGN / "align.pkl").exists():
    shutil.copy(ALIGN / "align.pkl", TEMPLATE)

def look_pose(pos, pitch, yaw=0.0):
    """OpenCV 约定的相机位姿(基座系)。yaw=0 面朝 -x(朝机器人),pitch=下俯角。"""
    th, ps = np.radians(pitch), np.radians(yaw)
    z = np.array([-np.cos(th) * np.cos(ps), -np.cos(th) * np.sin(ps), -np.sin(th)])
    x = np.cross(z, [0, 0, 1]); x /= np.linalg.norm(x); y = np.cross(z, x)
    T = np.eye(4); T[:3, :3] = np.c_[x, y, z]; T[:3, 3] = pos; return T

_FRONT = re.compile(r"(cam_pos = torch\.from_numpy\(np\.array\()\[([^\]]*)\](\)\.reshape\(1, 3\)\).*?np\.deg2rad\()\[([^\]]*)\]", re.S)
def read_front():
    m = _FRONT.search(RR.read_text())
    pw = np.array([float(v) for v in m.group(2).split(",")]); eul = [float(v) for v in m.group(4).split(",")]
    f = (R.from_euler("XYZ", eul, degrees=True).as_matrix() @ FLIP)[:, 2]
    return pw - BASE_TO_WORLD, float(np.degrees(np.arcsin(-f[2]))), float(np.degrees(np.arctan2(-f[1], -f[0])))
def write_front(pos, pitch, yaw):
    T = look_pose(pos, pitch, yaw)
    pw = np.round(T[:3, 3] + BASE_TO_WORLD, 4).tolist()
    eul = np.round(R.from_matrix(T[:3, :3] @ FLIP).as_euler("XYZ", degrees=True), 3).tolist()
    txt = RR.read_text()
    RR.write_text(_FRONT.sub(lambda m: f"{m.group(1)}{pw}{m.group(3)}{eul}", txt, count=1))
def write_intrinsics(name, k):
    block = (f"{name}_435_camera_matrix = np.array(\n    [\n        [{k['fx']:.8f}, 0.0, {k['cx']:.8f}],\n"
             f"        [0.0, {k['fy']:.8f}, {k['cy']:.8f}],\n        [0.0, 0.0, 1.0],\n    ]\n)")
    txt = RR.read_text(); pat = re.compile(rf"{name}_435_camera_matrix = np\.array\(\n    \[\n.*?\n    \]\n\)", re.S)
    assert pat.search(txt), name; RR.write_text(pat.sub(lambda m: block, txt, count=1))

def edges(rgb):
    return cv2.Canny(cv2.GaussianBlur(cv2.cvtColor(rgb, cv2.COLOR_RGB2GRAY), (5, 5), 0), 40, 120) > 0
def clean_mask(m):
    m = ndimage.binary_opening(m, iterations=2)
    lab, n = ndimage.label(m)
    if n: m = lab == (np.argmax(ndimage.sum(m, lab, range(1, n + 1))) + 1)
    return ndimage.binary_fill_holes(m)
print("前视外参(渲染脚本当前值):  位置 %s m  俯角 %.1f°  偏航 %.1f°" % ((lambda p: (np.round(p[0], 3).tolist(), p[1], p[2]))(read_front())))

## 2. 生成参考 pkl,并写入内参(每次拷来新的 `real_state.json` 后运行)

- 关节角和夹爪宽度用**实测值**,速度置零,3 帧都设成同一个静止状态。
- 零件和挡条全部移出画面(放到 20 m 外)。
- 两台相机的内参自动写进渲染脚本。渲染器只用到 fx、fy,用不到 cx、cy。

In [ ]:
st = json.loads((ALIGN / "real_state.json").read_text())
q = np.array(st["joint_positions"], dtype=np.float32); w = float(st["gripper_width"])
print("拍摄时间:", st.get("time"), "  与指令差(度):", np.round(np.degrees(q - np.array(st["commanded"])), 2).tolist())
print(f"夹爪宽度 {w * 100:.2f} cm")

d = unpickle_data(str(TEMPLATE))
d["observations"], d["actions"] = d["observations"][:3], d["actions"][:2]
for k in ("rewards", "skills"):
    if k in d: d[k] = d[k][:2]
n_poses = len(np.asarray(d["observations"][0]["parts_poses"])) // 7
far = np.asarray(d["observations"][0]["parts_poses"], dtype=np.float32).reshape(-1, 7).copy()
far[:, :3] = [[20.0 + i, 0.0, 0.0] for i in range(n_poses)]     # AprilTag 系 x = 基座系 y:放到 20 m 外
for o in d["observations"]:
    rs = o["robot_state"]
    rs["joint_positions"] = q.copy()
    rs["joint_velocities"] = np.zeros(7, dtype=np.float32)
    rs["gripper_width"] = np.array([w], dtype=np.float32)
    rs["gripper_finger_1_pos"] = rs["gripper_finger_2_pos"] = np.array([w / 2], dtype=np.float32)
    o["parts_poses"] = far.reshape(-1).copy()
shutil.rmtree(LOAD, ignore_errors=True); LOAD.mkdir(parents=True)
pickle.dump(d, open(LOAD / "align.pkl", "wb"))

for name in ("front", "wrist"):
    write_intrinsics(name, st["intrinsics"][name])
    print(f"{name} 内参已写入: fx={st['intrinsics'][name]['fx']:.2f} fy={st['intrinsics'][name]['fy']:.2f}")

real_front = cv2.cvtColor(cv2.imread(str(ALIGN / "real_front.png")), cv2.COLOR_BGR2RGB)
real_wrist = cv2.cvtColor(cv2.imread(str(ALIGN / "real_wrist.png")), cv2.COLOR_BGR2RGB)
fig, ax = plt.subplots(1, 2, figsize=(12, 4.5)); ax[0].imshow(real_front); ax[1].imshow(real_wrist)
ax[0].set_title("real front"); ax[1].set_title("real wrist"); [a.axis("off") for a in ax]; plt.show()

## 3. 前视外参

用基座系表示:位置(米)、俯角(向下为正,度)、偏航(面朝机器人为 0,度)。
不改就保持当前值。第 5 步会打印建议值,可以直接照填。

In [ ]:
pos, pitch, yaw = read_front()
print(f"当前: pos={np.round(pos, 3).tolist()}  pitch={pitch:.2f}  yaw={yaw:.2f}")
# ====== 改这里(取消注释再改)======
# pos   = [1.13, 0.00, 0.415]
# pitch = 26.0
# yaw   = 0.0
# ==================================
write_front(pos, pitch, yaw)
print("已写入渲染脚本:", *[f"{v}" for v in (np.round(pos, 3).tolist(), round(pitch, 2), round(yaw, 2))])

## 4. 渲染(约 1 分钟)

In [ ]:
shutil.rmtree(OUT, ignore_errors=True)
# resip 内核导入 IsaacGym/OpenCV 时会设置 CARB_APP_PATH、LD_LIBRARY_PATH 等,会让 Isaac Sim 启动失败,所以去掉
env = {k: v for k, v in os.environ.items()
       if k not in ("CARB_APP_PATH", "LD_LIBRARY_PATH", "PYTHONHOME") and not k.startswith(("GYM_", "QT_QPA_"))}
env.update(PYTHONPATH=f"{REPO}:{REPO}/furniture-bench", RARL_SOURCE_DIR=f"{REPO}/src",
           CONDA_PREFIX=str(ISAAC_PY.parents[1]), PATH=f"{ISAAC_PY.parent}:{os.environ['PATH']}")
cmd = [str(ISAAC_PY), "-m", "src.sim2real.isaac_lab_rerender", "--headless", "--enable_cameras", "-i", "0",
       "--sub-steps", "3", "--load-dir", str(LOAD), "--furniture", "one_leg", "--num-parts", "5",
       "--save", "--save-dir", str(OUT), "--robot", "fr3", "--robot-mask"]
r = subprocess.run(cmd, cwd=REPO, env=env, capture_output=True, text=True)
log = r.stdout + r.stderr
print("\n".join(l for l in log.splitlines() if l.startswith(("[robot]", "Saving new pickle"))))
pkls = sorted(OUT.glob("*.pkl"))
if not pkls: print(log[-3000:]); raise RuntimeError("渲染失败,见上面的日志")

## 5. 对比(只看机器人)

- **前视**:绿色是真实图的边缘,红色是仿真机器人的边缘(只取机器人掩码内),黄色是两者重合的地方。
- 打印"仿真要平移/缩放多少才能对上真实图",并换算成外参的建议值。
- **达标**:建议平移在 ±2 px 以内,缩放误差在 1% 以内。平均距离只作参考,因为真实 FR3 和渲染的外观不完全一样。1° 大约等于 10 px。
- **腕部**:只叠图检查,不调。

In [ ]:
o = unpickle_data(str(pkls[-1]))["observations"][0]
sim_f, sim_w = np.asarray(o["color_image3"]).astype(np.uint8), np.asarray(o["color_image1"]).astype(np.uint8)
mask = clean_mask(o["robot_mask_front"])
sim_e = edges(sim_f) & ndimage.binary_dilation(mask, iterations=2)       # 仿真机器人边缘(掩码含阴影,所以只取 Canny 边)
real_e = edges(real_front)
dt = cv2.distanceTransform((~real_e).astype(np.uint8), cv2.DIST_L2, 3)
ys, xs = np.nonzero(sim_e); cy0, cx0 = ys.mean(), xs.mean()
def score(s, tx, ty):
    X, Y = (cx0 + (xs - cx0) * s + tx).round().astype(int), (cy0 + (ys - cy0) * s + ty).round().astype(int)
    ok = (X >= 0) & (X < 640) & (Y >= 0) & (Y < 480)
    return np.mean(np.minimum(dt[Y[ok], X[ok]], 12)) if ok.mean() > 0.8 else 99
best = min((score(s, tx, ty), s, tx, ty) for s in np.arange(0.90, 1.105, 0.01)
           for tx in range(-40, 41, 2) for ty in range(-40, 41, 2))
best = min((score(s, tx, ty), s, tx, ty) for s in np.arange(best[1] - 0.01, best[1] + 0.011, 0.005)
           for tx in range(best[2] - 2, best[2] + 3) for ty in range(best[3] - 2, best[3] + 3))
c_now, (c_best, s, tx, ty) = score(1, 0, 0), best
F = (st["intrinsics"]["front"]["fx"] + st["intrinsics"]["front"]["fy"]) / 2
pos, pitch, yaw = read_front()
d_yaw = np.degrees(np.arctan(tx / F)) / np.cos(np.radians(pitch))   # 偏航增大 -> 画面内容右移
d_pitch = -np.degrees(np.arctan(ty / F))                            # 俯角增大 -> 画面内容上移
T = look_pose(pos, pitch, yaw); D = np.linalg.norm(T[:3, 3] - [0.3, 0.0, 0.3])   # 到机器人中部的距离
new_pos = T[:3, 3] + T[:3, 2] * (D - D / s)                         # 真实更大(s>1) -> 沿视线靠近
SUGGEST = dict(pos=np.round(new_pos, 3).tolist(), pitch=round(pitch + d_pitch, 2), yaw=round(yaw + d_yaw, 2))
print(f"机器人边缘平均距离: 现在 {c_now:.2f} px  ->  平移/缩放后 {c_best:.2f} px")
print(f"仿真需要: 平移 ({tx:+d}, {ty:+d}) px, 缩放 x{s:.3f}")
ok = abs(tx) <= 2 and abs(ty) <= 2 and abs(s - 1) <= 0.01
print("达标 ✓" if ok else f"建议(第 3 步照填,先改角度,大小还差再改位置):\n  pitch = {SUGGEST['pitch']}\n  yaw   = {SUGGEST['yaw']}\n  pos   = {SUGGEST['pos']}")

ov = (0.45 * real_front).astype(np.uint8); ov[real_e] = (0, 255, 0); ov[sim_e] = (255, 0, 0); ov[real_e & sim_e] = (255, 255, 0)
cv2.rectangle(ov, (96, 16), (544, 464), (0, 255, 255), 1)
ow = cv2.addWeighted(real_wrist, 0.5, sim_w, 0.5, 0); ow[edges(sim_w)] = (255, 0, 0)
fig, ax = plt.subplots(1, 2, figsize=(16, 6))
ax[0].imshow(ov); ax[0].set_title("FRONT: green = real edges, red = sim robot edges")
ax[1].imshow(ow); ax[1].set_title("WRIST (check only): 50/50 blend, red = sim edges")
[a.axis("off") for a in ax]; plt.show()
cv2.imwrite(str(ALIGN / "overlay_front.png"), cv2.cvtColor(ov, cv2.COLOR_RGB2BGR))
cv2.imwrite(str(ALIGN / "overlay_wrist.png"), cv2.cvtColor(ow, cv2.COLOR_RGB2BGR))

**腕部对不上怎么办**:前视达标说明关节角是对的。这时腕部如果还差很多,问题在硬件安装,不在参数,请检查:
- 支架是不是装在手爪背面(−x 侧),镜头朝向指尖;
- 装得是否到位、牢固;
- 夹爪宽度是否和 `real_state.json` 里记录的一致。

## 6. 保存基准(达标后)
之后每天开工前,用同一个参考姿态拍照,和这里存下的图对比,检查相机有没有被碰动(§9.1)。

In [ ]:
import time
B = ALIGN / f"baseline_{time.strftime('%Y%m%d_%H%M')}"; B.mkdir()
for f in ["real_state.json", "real_front.png", "real_wrist.png", "overlay_front.png", "overlay_wrist.png"]:
    shutil.copy(ALIGN / f, B / f)
pos, pitch, yaw = read_front()
(B / "front_extrinsics.json").write_text(json.dumps(dict(pos_base=np.round(pos, 4).tolist(), pitch=pitch, yaw=yaw), indent=1))
print("已保存到", B)